In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
CATALOG_SCHEMA = "workspace.stocks"
FACT = f"{CATALOG_SCHEMA}.fact_daily_price"
DIM = f"{CATALOG_SCHEMA}.dim_company"


In [0]:
fact = spark.table(FACT)
w20 = Window.partitionBy('ticker').orderBy(F.col('trade_date')).rowsBetween(-19,0)
with_ma = fact.withColumn("ma20", F.avg("close").over(w20))

w_last = Window.partitionBy("ticker").orderBy(F.col("trade_date").desc())
latest = with_ma.withColumn("rn", F.row_number().over(w_last)).filter(F.col("rn") == 1)

trend = (latest
    .withColumn("trend_status",
                F.when(F.col("close") >= F.col("ma20"), "above_avg").otherwise("below_avg"))
    .withColumnRenamed("trade_date", "effective_date")             
    .select("ticker", "effective_date", "close", "ma20", "trend_status"))
trend.show()


+------+--------------+--------+------------+------------+
|ticker|effective_date|   close|        ma20|trend_status|
+------+--------------+--------+------------+------------+
|  AAPL|    2026-11-03|331.9000|342.26825000|   below_avg|
|   AMD|    2026-11-03|483.9000|553.91900000|   below_avg|
|  AMZN|    2026-11-03|250.6900|253.82200000|   below_avg|
| GOOGL|    2026-11-03|280.0300|327.14975000|   below_avg|
|   IBM|    2026-11-03|250.0400|241.85900000|   above_avg|
|  MSFT|    2026-11-03|520.9900|529.66250000|   below_avg|
|  NVDA|    2026-11-03|206.6900|217.99000000|   below_avg|
|  TSLA|    2026-11-03|398.5000|382.84300000|   above_avg|
+------+--------------+--------+------------+------------+



In [0]:
companies = spark.table(f"{CATALOG_SCHEMA}.bronze_companies").select("ticker", "name", "sector", "industry")

updates = (companies
    .join(trend.select("ticker", "effective_date", "trend_status"), on="ticker", how="inner")
    .select("ticker", "name", "sector", "industry", "trend_status", "effective_date"))

updates.createOrReplaceTempView("updates")
updates.show()

+------+--------------------+--------------------+--------------------+------------+--------------+
|ticker|                name|              sector|            industry|trend_status|effective_date|
+------+--------------------+--------------------+--------------------+------------+--------------+
|   IBM|INTERNATIONAL BUS...|          Technology|Information Techn...|   above_avg|    2026-11-03|
|  TSLA|         TESLA, INC.|   Consumer Cyclical|  Auto Manufacturers|   above_avg|    2026-11-03|
|  NVDA|  NVIDIA CORPORATION|          Technology|      Semiconductors|   below_avg|    2026-11-03|
|  AAPL|          APPLE INC.|          Technology|Consumer Electronics|   below_avg|    2026-11-03|
|   AMD|ADVANCED MICRO DE...|          Technology|      Semiconductors|   below_avg|    2026-11-03|
| GOOGL|       ALPHABET INC.|Communication Ser...|Internet Content ...|   below_avg|    2026-11-03|
|  MSFT|MICROSOFT CORPORA...|          Technology|Software - Infras...|   below_avg|    2026-11-03|


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM} (
  ticker STRING, name STRING, sector STRING, industry STRING, trend_status STRING,
  valid_from DATE, valid_to DATE, is_current BOOLEAN
) USING DELTA
""")

DataFrame[]

In [0]:
display(spark.sql(f"""
MERGE INTO {DIM} AS t
USING updates AS s
ON t.ticker = s.ticker AND t.is_current = true

WHEN MATCHED AND t.valid_from = s.effective_date AND (
       NOT (t.name         <=> s.name)
    OR NOT (t.sector       <=> s.sector)
    OR NOT (t.industry     <=> s.industry)
    OR NOT (t.trend_status <=> s.trend_status)
) THEN UPDATE SET
       t.name = s.name, t.sector = s.sector, t.industry = s.industry, t.trend_status = s.trend_status

WHEN MATCHED AND t.valid_from < s.effective_date AND (
       NOT (t.name         <=> s.name)
    OR NOT (t.sector       <=> s.sector)
    OR NOT (t.industry     <=> s.industry)
    OR NOT (t.trend_status <=> s.trend_status)
) THEN UPDATE SET
       t.is_current = false,
       t.valid_to   = date_sub(s.effective_date, 1)
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
display(spark.sql(f"""
INSERT INTO {DIM}
SELECT s.ticker, s.name, s.sector, s.industry, s.trend_status,
       s.effective_date      AS valid_from,
       DATE'9999-12-31'      AS valid_to,
       true                AS is_current
FROM updates s
LEFT ANTI JOIN {DIM} t
  ON t.ticker = s.ticker AND t.is_current = true
"""))



num_affected_rows,num_inserted_rows
2,2


In [0]:
dim = spark.table(DIM)
print("total cards:", dim.count())

bad = (dim.filter(F.col("is_current") == True)
          .groupBy("ticker").count()
          .filter(F.col("count") > 1))
print("tickers with more than 1 current card:", bad.count())   
impossible = dim.filter(F.col("valid_to") < F.col("valid_from")).count()
print("cards that end before they start:", impossible)
dim.orderBy("ticker", "valid_from").show(truncate=False)

total cards: 10
tickers with more than 1 current card: 0
cards that end before they start: 0
+------+-------------------------------------------+----------------------+-------------------------------+------------+----------+----------+----------+
|ticker|name                                       |sector                |industry                       |trend_status|valid_from|valid_to  |is_current|
+------+-------------------------------------------+----------------------+-------------------------------+------------+----------+----------+----------+
|AAPL  |APPLE INC.                                 |Technology            |Consumer Electronics           |above_avg   |2026-10-28|2026-11-02|false     |
|AAPL  |APPLE INC.                                 |Technology            |Consumer Electronics           |below_avg   |2026-11-03|9999-12-31|true      |
|AMD   |ADVANCED MICRO DEVICES, INC.               |Technology            |Semiconductors                 |below_avg   |2026-10-28|9999-1